In [1]:
import os
os.chdir("..")

import sys
sys.path.append("src")

import pandas as pd

from wildfire_gnn.pipelines.cnn_baseline_pipeline import CNNBaselinePipeline
from wildfire_gnn.utils.config import load_yaml_config

In [2]:
config = load_yaml_config("configs/cnn_baseline_config.yaml")
pipeline = CNNBaselinePipeline(config)

In [3]:
from pathlib import Path
import rasterio

paths = list(Path("data/interim/aligned").glob("*.img"))

for p in paths:
    with rasterio.open(p) as src:
        print(p.name, src.shape, src.dtypes, src.nodata)

Burn_Prob.img (7597, 7555) ('float32',) -9999.0
CFL.img (7597, 7555) ('float32',) -9999.0
FSP_Index.img (7597, 7555) ('float32',) -9999.0
Fuel_Models.img (7597, 7555) ('uint8',) 0.0
Ignition_Prob.img (7597, 7555) ('float32',) -9999.0
Struct_Exp_Index.img (7597, 7555) ('float32',) -9999.0


In [4]:
results_df = pipeline.run()
results_df

2026-04-02 18:32:57 | INFO | cnn_baseline_pipeline | Stratified bin [0.0, 0.01): available=5356465 sampled=40000
2026-04-02 18:32:58 | INFO | cnn_baseline_pipeline | Stratified bin [0.01, 0.05): available=4727847 sampled=40000
2026-04-02 18:32:59 | INFO | cnn_baseline_pipeline | Stratified bin [0.05, 0.1): available=1160219 sampled=40000
2026-04-02 18:32:59 | INFO | cnn_baseline_pipeline | Stratified bin [0.1, 0.25): available=507855 sampled=40000
2026-04-02 18:32:59 | INFO | cnn_baseline_pipeline | Stratified bin [0.25, 1.0): available=16 sampled=16
2026-04-02 18:33:04 | INFO | cnn_baseline_pipeline | Applied stratified subsampling: kept 200000 / 11752402 patches
2026-04-02 18:33:06 | INFO | cnn_baseline_pipeline | Saved patch metadata to data/processed/cnn_patch_metadata.csv
2026-04-02 18:33:06 | INFO | cnn_baseline_pipeline | CNN dataset stats: original_samples=11752402 sampled_samples=200000 target_min=0.000000 target_max=0.250882 target_mean=0.052204 target_std=0.053710
2026-04-02

,model,split_type,data_split,rmse,mae,r2,pearson,spearman
0,cnn,random,val,0.022576,0.015515,0.819484,0.912219,0.931534
1,cnn,random,test,0.022593,0.015526,0.821805,0.914212,0.932958
2,cnn,spatial,val,0.025600,0.020139,0.705151,0.846233,0.874101
3,cnn,spatial,test,0.018484,0.015531,0.487520,0.848473,0.800038


In [5]:
metrics_df = pd.read_csv("reports/tables/cnn_metrics.csv")
metrics_df

,model,split_type,data_split,rmse,mae,r2,pearson,spearman
0,cnn,random,val,0.022576,0.015515,0.819484,0.912219,0.931534
1,cnn,random,test,0.022593,0.015526,0.821805,0.914212,0.932958
2,cnn,spatial,val,0.025600,0.020139,0.705151,0.846233,0.874101
3,cnn,spatial,test,0.018484,0.015531,0.487520,0.848473,0.800038


In [6]:
cnn_random_preds = pd.read_csv("reports/tables/cnn_random_test_predictions.csv")
cnn_random_preds.head()

,row_index,col_index,y_true,y_pred,residual
0,4756,2197,0.088232,0.085811,0.002421
1,3728,2555,0.175064,0.089820,0.085244
2,1527,1837,0.122934,0.092039,0.030895
3,4296,2558,0.064533,0.056204,0.008329
4,1566,2981,0.047159,0.078844,-0.031685


In [7]:
cnn_spatial_preds = pd.read_csv("reports/tables/cnn_spatial_test_predictions.csv")
cnn_spatial_preds.head()

,row_index,col_index,y_true,y_pred,residual
0,4890,1812,0.019513,0.030981,-0.011469
1,1587,2502,0.033439,0.056645,-0.023205
2,846,3546,0.003566,0.031047,-0.027481
3,2531,1205,0.005752,0.022612,-0.016861
4,2201,1277,0.000667,0.017680,-0.017013
